In [ ]:
# Import libraries

import csv
import json
import math
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.init import trunc_normal_
from torch.utils.data import Dataset, DataLoader

cv2.setNumThreads(0)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. Select an environment with CUDA-enabled PyTorch.")

DEVICE = torch.device("cuda:0")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(DEVICE))
import hashlib
import shutil
from datetime import datetime
import copy


In [ ]:
# Configure paths and experiment

EXPERIMENT_NAME = "V2"

# Update these paths before running the notebook.
DATA_ROOT = Path("data") / EXPERIMENT_NAME
OUTPUT_DIR = Path("results") / "segformer_calibrated" / EXPERIMENT_NAME
PRETRAINED_CHECKPOINT = Path("pretrained") / "mit_b2_20220624-66e8bf70.pth"

BATCH_SIZE = 2
EVAL_BATCH_SIZE = 1
EPOCHS = 50
LEARNING_RATE = 1e-4
THRESHOLD = 0.5
TRAINING_SEED = 145
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()
SAVE_EVERY_EPOCH = True

EXPECTED_COUNTS = {"Train": 967, "Validation": 276, "Test": 138}
if EXPERIMENT_NAME != "V2":
    raise ValueError("This notebook supports V2 only.")

assert DATA_ROOT.is_dir(), f"Dataset directory not found: {DATA_ROOT}"
assert PRETRAINED_CHECKPOINT.is_file(), f"Pretrained checkpoint not found: {PRETRAINED_CHECKPOINT}"
print("Experiment:", EXPERIMENT_NAME)
print("Dataset:", DATA_ROOT)
print("Output:", OUTPUT_DIR)
print("Pretrained:", PRETRAINED_CHECKPOINT)


In [ ]:
# Define the dataset

IMAGE_EXTENSIONS = {".tif", ".tiff", ".png", ".jpg", ".jpeg"}
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class BuildingPatchDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.images = {
            path.name: path for path in self.image_dir.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        }
        self.masks = {
            path.name: path for path in self.mask_dir.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        }
        if self.images.keys() != self.masks.keys():
            only_images = sorted(self.images.keys() - self.masks.keys())[:5]
            only_masks = sorted(self.masks.keys() - self.images.keys())[:5]
            raise ValueError(
                f"Image and mask filenames do not match. "
                f"Images without masks: {only_images}; masks without images: {only_masks}"
            )
        self.names = sorted(self.images)
        if not self.names:
            raise ValueError(f"No images found: {self.image_dir}")

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        image_bgr = cv2.imread(str(self.images[name]), cv2.IMREAD_COLOR)
        raw_mask = cv2.imread(str(self.masks[name]), cv2.IMREAD_GRAYSCALE)
        if image_bgr is None or raw_mask is None:
            raise RuntimeError(f"Cannot read file: {name}")
        if image_bgr.shape[:2] != raw_mask.shape:
            raise ValueError(f"Image and mask dimensions do not match: {name}")

        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        image = (image - IMAGENET_MEAN) / IMAGENET_STD
        image = torch.from_numpy(np.ascontiguousarray(image.transpose(2, 0, 1)))

        # Preserve the reference positive-target convention: raw_mask == 0.
        mask = (raw_mask == 0).astype(np.float32)[None, ...]
        mask = torch.from_numpy(mask)
        return image, mask


In [ ]:
# Check image and mask pairs

DATASETS = {EXPERIMENT_NAME: {}}
for split, expected in EXPECTED_COUNTS.items():
    dataset = BuildingPatchDataset(
        DATA_ROOT / split / "Image", DATA_ROOT / split / "Mask"
    )
    if len(dataset) != expected:
        raise ValueError(
            f"{split}: expected {expected} patches, found {len(dataset)}."
        )
    DATASETS[EXPERIMENT_NAME][split] = dataset
    print(f"{EXPERIMENT_NAME}/{split}: {len(dataset)} patches")


In [ ]:
# Define the SegFormer model
# Architecture reference: https://github.com/NVlabs/SegFormer

from functools import partial

def to_2tuple(value):
    return value if isinstance(value, tuple) else (value, value)


class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = float(drop_prob)

    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        return x * random_tensor.floor() / keep_prob


class DWConv(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, 3, 1, 1, bias=True, groups=dim)

    def forward(self, x, height, width):
        batch, _, channels = x.shape
        x = x.transpose(1, 2).reshape(batch, channels, height, width)
        x = self.dwconv(x)
        return x.flatten(2).transpose(1, 2)


class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.0):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.dwconv = DWConv(hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)
        elif isinstance(module, nn.LayerNorm):
            nn.init.constant_(module.bias, 0)
            nn.init.constant_(module.weight, 1.0)
        elif isinstance(module, nn.Conv2d):
            fan_out = module.kernel_size[0] * module.kernel_size[1] * module.out_channels
            fan_out //= module.groups
            nn.init.normal_(module.weight, 0, math.sqrt(2.0 / fan_out))
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)

    def forward(self, x, height, width):
        x = self.fc1(x)
        x = self.dwconv(x, height, width)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        return self.drop(x)


class Attention(nn.Module):
    def __init__(self, dim, num_heads, sr_ratio):
        super().__init__()
        if dim % num_heads != 0:
            raise ValueError("Embedding dimension must be divisible by number of heads")
        self.dim = dim
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.q = nn.Linear(dim, dim, bias=True)
        self.kv = nn.Linear(dim, dim * 2, bias=True)
        self.attn_drop = nn.Dropout(0.0)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(0.0)
        self.sr_ratio = sr_ratio
        if sr_ratio > 1:
            self.sr = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio)
            self.norm = nn.LayerNorm(dim)
        self.apply(Mlp._init_weights)

    def forward(self, x, height, width):
        batch, tokens, channels = x.shape
        q = self.q(x).reshape(batch, tokens, self.num_heads, channels // self.num_heads)
        q = q.permute(0, 2, 1, 3)
        if self.sr_ratio > 1:
            reduced = x.permute(0, 2, 1).reshape(batch, channels, height, width)
            reduced = self.sr(reduced).reshape(batch, channels, -1).permute(0, 2, 1)
            reduced = self.norm(reduced)
        else:
            reduced = x
        kv = self.kv(reduced).reshape(
            batch, -1, 2, self.num_heads, channels // self.num_heads
        ).permute(2, 0, 3, 1, 4)
        key, value = kv[0], kv[1]
        attention = (q @ key.transpose(-2, -1)) * self.scale
        attention = self.attn_drop(attention.softmax(dim=-1))
        x = (attention @ value).transpose(1, 2).reshape(batch, tokens, channels)
        return self.proj_drop(self.proj(x))


class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio, drop_path, sr_ratio):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = Attention(dim, num_heads=num_heads, sr_ratio=sr_ratio)
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = Mlp(dim, hidden_features=int(dim * mlp_ratio))
        self.apply(Mlp._init_weights)

    def forward(self, x, height, width):
        x = x + self.drop_path(self.attn(self.norm1(x), height, width))
        return x + self.drop_path(self.mlp(self.norm2(x), height, width))


class OverlapPatchEmbed(nn.Module):
    def __init__(self, patch_size, stride, in_chans, embed_dim):
        super().__init__()
        patch_size = to_2tuple(patch_size)
        self.proj = nn.Conv2d(
            in_chans,
            embed_dim,
            kernel_size=patch_size,
            stride=stride,
            padding=(patch_size[0] // 2, patch_size[1] // 2),
        )
        self.norm = nn.LayerNorm(embed_dim)
        self.apply(Mlp._init_weights)

    def forward(self, x):
        x = self.proj(x)
        _, _, height, width = x.shape
        x = self.norm(x.flatten(2).transpose(1, 2))
        return x, height, width


class MixVisionTransformerB2(nn.Module):
    """NVlabs MiT-B2 encoder; feature outputs are at strides 4, 8, 16, 32."""

    def __init__(self, drop_path_rate=0.1):
        super().__init__()
        embed_dims = [64, 128, 320, 512]
        num_heads = [1, 2, 5, 8]
        depths = [3, 4, 6, 3]
        sr_ratios = [8, 4, 2, 1]
        mlp_ratios = [4, 4, 4, 4]
        self.depths = depths
        self.patch_embed1 = OverlapPatchEmbed(7, 4, 3, embed_dims[0])
        self.patch_embed2 = OverlapPatchEmbed(3, 2, embed_dims[0], embed_dims[1])
        self.patch_embed3 = OverlapPatchEmbed(3, 2, embed_dims[1], embed_dims[2])
        self.patch_embed4 = OverlapPatchEmbed(3, 2, embed_dims[2], embed_dims[3])
        drop_rates = torch.linspace(0, drop_path_rate, sum(depths)).tolist()
        offset = 0
        for stage in range(4):
            blocks = nn.ModuleList(
                [
                    Block(
                        dim=embed_dims[stage],
                        num_heads=num_heads[stage],
                        mlp_ratio=mlp_ratios[stage],
                        drop_path=drop_rates[offset + index],
                        sr_ratio=sr_ratios[stage],
                    )
                    for index in range(depths[stage])
                ]
            )
            setattr(self, f"block{stage + 1}", blocks)
            setattr(self, f"norm{stage + 1}", nn.LayerNorm(embed_dims[stage], eps=1e-6))
            offset += depths[stage]

    def forward(self, x):
        outputs = []
        batch = x.shape[0]
        for stage in range(1, 5):
            patch_embed = getattr(self, f"patch_embed{stage}")
            blocks = getattr(self, f"block{stage}")
            norm = getattr(self, f"norm{stage}")
            x, height, width = patch_embed(x)
            for block in blocks:
                x = block(x, height, width)
            x = norm(x).reshape(batch, height, width, -1).permute(0, 3, 1, 2).contiguous()
            outputs.append(x)
        return outputs


class MLPEmbedding(nn.Module):
    def __init__(self, input_dim, embed_dim):
        super().__init__()
        self.proj = nn.Linear(input_dim, embed_dim)

    def forward(self, x):
        return self.proj(x.flatten(2).transpose(1, 2))


class SegFormerHeadBinary(nn.Module):
    """NVlabs SegFormer MLP decoder adapted for single-channel segmentation."""

    def __init__(self, embed_dim=256, dropout_ratio=0.1):
        super().__init__()
        channels = [64, 128, 320, 512]
        self.linear_c1 = MLPEmbedding(channels[0], embed_dim)
        self.linear_c2 = MLPEmbedding(channels[1], embed_dim)
        self.linear_c3 = MLPEmbedding(channels[2], embed_dim)
        self.linear_c4 = MLPEmbedding(channels[3], embed_dim)
        self.linear_fuse = nn.Sequential(
            nn.Conv2d(embed_dim * 4, embed_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.ReLU(inplace=True),
        )
        self.dropout = nn.Dropout2d(dropout_ratio)
        self.linear_pred = nn.Conv2d(embed_dim, 1, kernel_size=1)
        self.apply(Mlp._init_weights)

    @staticmethod
    def _project(embedding, feature):
        batch, _, height, width = feature.shape
        return embedding(feature).transpose(1, 2).reshape(batch, -1, height, width)

    def forward(self, features):
        c1, c2, c3, c4 = features
        target_size = c1.shape[-2:]
        projected = [
            self._project(self.linear_c1, c1),
            self._project(self.linear_c2, c2),
            self._project(self.linear_c3, c3),
            self._project(self.linear_c4, c4),
        ]
        projected[1:] = [
            F.interpolate(feature, size=target_size, mode="bilinear", align_corners=False)
            for feature in projected[1:]
        ]
        fused = self.linear_fuse(torch.cat(projected, dim=1))
        return self.linear_pred(self.dropout(fused))


class SegFormerBinary(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = MixVisionTransformerB2()
        self.decode_head = SegFormerHeadBinary()

    def forward(self, images):
        features = self.backbone(images)
        logits = self.decode_head(features)
        return F.interpolate(logits, size=images.shape[-2:], mode="bilinear", align_corners=False)


def load_pretrained_mit_b2(backbone, checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Pretrained checkpoint not found: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    state = checkpoint
    if isinstance(checkpoint, dict):
        for container_key in ("state_dict", "model", "model_state_dict"):
            if container_key in checkpoint and isinstance(checkpoint[container_key], dict):
                state = checkpoint[container_key]
                break
    if not isinstance(state, dict):
        raise TypeError("Checkpoint does not contain a state dictionary")

    normalized_state = {}
    for original_key, value in state.items():
        key = original_key
        for prefix in ("module.", "backbone."):
            if key.startswith(prefix):
                key = key[len(prefix):]
        normalized_state[key] = value

    target_state = backbone.state_dict()
    matched = {}

    def add(source_key, target_key, transform=lambda tensor: tensor):
        value = normalized_state.get(source_key)
        if value is None or target_key not in target_state:
            return
        value = transform(value)
        if tuple(value.shape) == tuple(target_state[target_key].shape):
            matched[target_key] = value

    # Convert current OpenMMLab MiT checkpoint names/layout to NVlabs MiT names.
    for stage, depth in enumerate(backbone.depths):
        stage_id = stage + 1
        source_patch = f"layers.{stage}.0."
        target_patch = f"patch_embed{stage_id}."
        for source_name, target_name in (
            ("projection.weight", "proj.weight"),
            ("projection.bias", "proj.bias"),
            ("norm.weight", "norm.weight"),
            ("norm.bias", "norm.bias"),
        ):
            add(source_patch + source_name, target_patch + target_name)

        add(f"layers.{stage}.2.weight", f"norm{stage_id}.weight")
        add(f"layers.{stage}.2.bias", f"norm{stage_id}.bias")

        for block_index in range(depth):
            source_block = f"layers.{stage}.1.{block_index}."
            target_block = f"block{stage_id}.{block_index}."
            for suffix in ("norm1.weight", "norm1.bias", "norm2.weight", "norm2.bias"):
                add(source_block + suffix, target_block + suffix)

            qkv_weight = normalized_state.get(source_block + "attn.attn.in_proj_weight")
            qkv_bias = normalized_state.get(source_block + "attn.attn.in_proj_bias")
            if qkv_weight is not None:
                q_weight, k_weight, v_weight = qkv_weight.chunk(3, dim=0)
                add(source_block + "attn.attn.in_proj_weight", target_block + "attn.q.weight",
                    lambda _unused, tensor=q_weight: tensor)
                add(source_block + "attn.attn.in_proj_weight", target_block + "attn.kv.weight",
                    lambda _unused, tensor=torch.cat((k_weight, v_weight), dim=0): tensor)
            if qkv_bias is not None:
                q_bias, k_bias, v_bias = qkv_bias.chunk(3, dim=0)
                add(source_block + "attn.attn.in_proj_bias", target_block + "attn.q.bias",
                    lambda _unused, tensor=q_bias: tensor)
                add(source_block + "attn.attn.in_proj_bias", target_block + "attn.kv.bias",
                    lambda _unused, tensor=torch.cat((k_bias, v_bias), dim=0): tensor)

            for source_name, target_name in (
                ("attn.attn.out_proj.weight", "attn.proj.weight"),
                ("attn.attn.out_proj.bias", "attn.proj.bias"),
                ("attn.sr.weight", "attn.sr.weight"),
                ("attn.sr.bias", "attn.sr.bias"),
                ("attn.norm.weight", "attn.norm.weight"),
                ("attn.norm.bias", "attn.norm.bias"),
                ("ffn.layers.0.bias", "mlp.fc1.bias"),
                ("ffn.layers.1.weight", "mlp.dwconv.dwconv.weight"),
                ("ffn.layers.1.bias", "mlp.dwconv.dwconv.bias"),
                ("ffn.layers.4.bias", "mlp.fc2.bias"),
            ):
                add(source_block + source_name, target_block + target_name)

            add(source_block + "ffn.layers.0.weight", target_block + "mlp.fc1.weight",
                lambda tensor: tensor.squeeze(-1).squeeze(-1))
            add(source_block + "ffn.layers.4.weight", target_block + "mlp.fc2.weight",
                lambda tensor: tensor.squeeze(-1).squeeze(-1))

    total_params = sum(parameter.numel() for parameter in backbone.parameters())
    matched_params = sum(target_state[key].numel() for key in matched)
    coverage = matched_params / total_params
    if coverage < 0.99:
        raise RuntimeError(
            f"Only {coverage:.1%} of MiT-B2 parameters matched; refusing to continue. "
            f"Matched {len(matched)}/{len(target_state)} state entries."
        )

    result = backbone.load_state_dict(matched, strict=False)
    print(f"Pretrained encoder tensors loaded: {len(matched)}/{len(target_state)}")
    print(f"Pretrained encoder parameter coverage: {coverage:.2%}")
    print(f"Unmatched encoder keys: {len(result.missing_keys)}")
    if result.missing_keys:
        print("Missing encoder keys:", result.missing_keys[:10])
    return coverage


In [ ]:
# Define Dice loss and patch metrics

def dice_loss_with_logits(logits, targets, eps=1e-7):
    probabilities = torch.sigmoid(logits).flatten(start_dim=1)
    targets = targets.flatten(start_dim=1)
    intersection = (probabilities * targets).sum(dim=1)
    denominator = probabilities.sum(dim=1) + targets.sum(dim=1)
    dice = (2.0 * intersection + eps) / (denominator + eps)
    return 1.0 - dice.mean()


@torch.no_grad()
def patch_metrics(logits, targets):
    predictions = torch.sigmoid(logits) >= THRESHOLD
    targets = targets >= 0.5
    predictions = predictions.flatten(start_dim=1)
    targets = targets.flatten(start_dim=1)

    tp = (predictions & targets).sum(dim=1).float()
    fp = (predictions & ~targets).sum(dim=1).float()
    fn = (~predictions & targets).sum(dim=1).float()
    tn = (~predictions & ~targets).sum(dim=1).float()
    eps = 1e-7

    return {
        "accuracy": (tp + tn) / (tp + tn + fp + fn + eps),
        "iou": tp / (tp + fp + fn + eps),
        "f1": (2 * tp) / (2 * tp + fp + fn + eps),
        "precision": tp / (tp + fp + eps),
        "recall": tp / (tp + fn + eps),
    }


In [ ]:
# Define the epoch routine

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

METRIC_NAMES = ("accuracy", "iou", "f1", "precision", "recall")


def run_epoch(model, loader, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    totals = {"loss": 0.0, **{name: 0.0 for name in METRIC_NAMES}}
    sample_count = 0

    for images, masks in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        batch_size = images.shape[0]

        if is_training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = dice_loss_with_logits(logits, masks)
            if is_training:
                loss.backward()
                optimizer.step()

        batch_metrics = patch_metrics(logits.detach(), masks)
        totals["loss"] += loss.detach().item() * batch_size
        for name, values in batch_metrics.items():
            totals[name] += values.sum().item()
        sample_count += batch_size

    if sample_count == 0:
        raise RuntimeError("The DataLoader is empty.")
    return {name: value / sample_count for name, value in totals.items()}


In [ ]:
# Create data loaders

def make_loaders(split_datasets):
    generator = torch.Generator().manual_seed(TRAINING_SEED)
    return {
        "Train": DataLoader(
            split_datasets["Train"], batch_size=BATCH_SIZE, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
            persistent_workers=False, generator=generator,
        ),
        "Validation": DataLoader(
            split_datasets["Validation"], batch_size=EVAL_BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
            persistent_workers=False,
        ),
        "Test": DataLoader(
            split_datasets["Test"], batch_size=EVAL_BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
            persistent_workers=False,
        ),
    }


In [ ]:
# Define checkpoint and calibration helpers

WEIGHTS_DIR = OUTPUT_DIR / "weights"
EPOCH_WEIGHTS_DIR = WEIGHTS_DIR / "epochs"
PREDICTION_DIR = OUTPUT_DIR / "prediction"
MASK_DIR = PREDICTION_DIR / "masks"
CONFUSION_DIR = OUTPUT_DIR / "confusion"
BEST_PATH = WEIGHTS_DIR / "best_model.pth"
FINAL_PATH = WEIGHTS_DIR / "final_model.pth"
HISTORY_PATH = OUTPUT_DIR / "history.json"
TRAIN_CSV_PATH = OUTPUT_DIR / "train_logs.csv"


def save_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temporary.replace(path)


def write_rows(path, rows):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def write_training_history(history):
    names = (("loss", "loss"), ("accuracy", "accuracy"), ("iou", "iou_score"),
             ("f1", "fscore"), ("precision", "precision"), ("recall", "recall"))
    rows = []
    for record in history:
        row = {"epoch": record["epoch"]}
        for key, column in names:
            row[column] = record["train"][key]
        for key, column in names:
            row["val_" + column] = record["validation"][key]
        for key, column in names:
            row["normal_val_" + column] = record["normal_validation"][key]
        rows.append(row)
    write_rows(TRAIN_CSV_PATH, rows)
    save_json(HISTORY_PATH, history)


def save_model_checkpoint(path, model, epoch, validation, best_val_iou):
    payload = {
        "epoch": epoch, "experiment": EXPERIMENT_NAME,
        "bn_calibrated": True, "validation": validation,
        "best_val_iou": best_val_iou,
        "model_state_dict": {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        },
    }
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)


def checkpoint_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


print("Weights:", WEIGHTS_DIR)
print("Prediction:", PREDICTION_DIR)
print("Confusion:", CONFUSION_DIR)


def recalibrate_bn_copy(source_model, calibration_loader):
    calibrated_model = copy.deepcopy(source_model).to(DEVICE)
    calibrated_model.eval()
    bn_layers = [module for module in calibrated_model.modules()
                 if isinstance(module, nn.BatchNorm2d)]
    if not bn_layers:
        raise RuntimeError("No BatchNorm2d layers found.")
    for module in bn_layers:
        module.reset_running_stats()
        module.momentum = None
        module.train()
    # Update only BN buffers; no backward pass or optimizer step.
    with torch.no_grad():
        for images, _ in calibration_loader:
            calibrated_model(images.to(DEVICE, non_blocking=True))
    calibrated_model.eval()
    return calibrated_model


In [ ]:
# Define the calibrated training loop

def train_calibrated_experiment():
    config = {
        "experiment": EXPERIMENT_NAME, "data_root": str(DATA_ROOT),
        "pretrained_checkpoint": str(PRETRAINED_CHECKPOINT),
        "epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "evaluation_batch_size": EVAL_BATCH_SIZE,
        "optimizer": "RMSprop", "learning_rate": LEARNING_RATE,
        "loss": "Dice loss", "threshold": THRESHOLD,
        "seed": TRAINING_SEED, "augmentation": "none",
        "bn_calibrated": True, "save_every_epoch": SAVE_EVERY_EPOCH,
        "bn_calibration_data": "Train", "bn_calibration_batch_size": BATCH_SIZE,
        "bn_calibration_shuffle": False,
        "torch_version": str(torch.__version__), "cuda_runtime": torch.version.cuda,
    }
    if FINAL_PATH.exists() and BEST_PATH.exists() and HISTORY_PATH.exists():
        old_config = json.loads((OUTPUT_DIR / "config.json").read_text(encoding="utf-8"))
        if any(old_config.get(key) != value for key, value in config.items()):
            raise RuntimeError("The saved run has different settings. Select another OUTPUT_DIR.")
        old_history = json.loads(HISTORY_PATH.read_text(encoding="utf-8"))
        final = torch.load(FINAL_PATH, map_location="cpu", weights_only=True)
        complete = len(old_history) == EPOCHS and final["epoch"] == EPOCHS
        if SAVE_EVERY_EPOCH:
            complete = complete and all(
                (EPOCH_WEIGHTS_DIR / f"epoch_{epoch:03d}.pth").is_file()
                for epoch in range(1, EPOCHS + 1)
            )
        if complete and final.get("bn_calibrated") is True:
            best_record = max(old_history, key=lambda row: row["validation"]["iou"])
            print(f"All {EPOCHS} epochs are complete; training will not be repeated.")
            print("Best epoch:", best_record["epoch"])
            return old_history
    if WEIGHTS_DIR.exists() and any(WEIGHTS_DIR.rglob("*.pth")):
        raise RuntimeError("An incomplete or different run exists. Select another OUTPUT_DIR; existing weights will not be overwritten.")
    for path in (WEIGHTS_DIR, EPOCH_WEIGHTS_DIR, PREDICTION_DIR, MASK_DIR, CONFUSION_DIR):
        path.mkdir(parents=True, exist_ok=True)
    if SAVE_EVERY_EPOCH and shutil.disk_usage(OUTPUT_DIR).free < 6 * 1024**3:
        raise RuntimeError("At least 6 GiB of free disk space is required for checkpoints and visualizations.")
    save_json(OUTPUT_DIR / "config.json", config)
    set_seed(TRAINING_SEED)
    loaders = make_loaders(DATASETS[EXPERIMENT_NAME])
    calibration_loader = DataLoader(
        DATASETS[EXPERIMENT_NAME]["Train"], batch_size=BATCH_SIZE,
        shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=False,
    )
    model = SegFormerBinary().to(DEVICE)
    load_pretrained_mit_b2(model.backbone, PRETRAINED_CHECKPOINT)
    optimizer = torch.optim.RMSprop(model.parameters(), lr=LEARNING_RATE)
    best_iou, history = -1.0, []
    for epoch in range(1, EPOCHS + 1):
        train_stats = run_epoch(model, loaders["Train"], optimizer)
        normal_val = run_epoch(model, loaders["Validation"])
        calibrated_model = recalibrate_bn_copy(model, calibration_loader)
        val_stats = run_epoch(calibrated_model, loaders["Validation"])
        history.append({"epoch": epoch, "train": train_stats,
                        "normal_validation": normal_val, "validation": val_stats})
        improved = val_stats["iou"] > best_iou
        if improved:
            best_iou = val_stats["iou"]
            save_model_checkpoint(BEST_PATH, calibrated_model, epoch, val_stats, best_iou)
        if SAVE_EVERY_EPOCH:
            save_model_checkpoint(
                EPOCH_WEIGHTS_DIR / f"epoch_{epoch:03d}.pth", calibrated_model, epoch, val_stats, best_iou
            )
        if epoch == EPOCHS:
            save_model_checkpoint(FINAL_PATH, calibrated_model, epoch, val_stats, best_iou)
        write_training_history(history)
        print(f"Epoch {epoch:02d}/{EPOCHS} | Train loss={train_stats['loss']:.4f}, "
              f"IoU={train_stats['iou']:.4f} | Normal val IoU={normal_val['iou']:.4f} | "
              f"Calibrated val loss={val_stats['loss']:.4f}, "
              f"IoU={val_stats['iou']:.4f}, F1={val_stats['f1']:.4f}")
        del calibrated_model
    del model, optimizer
    torch.cuda.empty_cache()
    print("Best checkpoint:", BEST_PATH)
    print("Final checkpoint:", FINAL_PATH)
    print("Train logs:", TRAIN_CSV_PATH)
    return history


In [ ]:
# Train the selected experiment

training_history = train_calibrated_experiment()


In [ ]:
# Plot training and validation metrics

with TRAIN_CSV_PATH.open(newline="", encoding="utf-8") as file:
    graph_rows = list(csv.DictReader(file))
graph_epochs = [int(row["epoch"]) for row in graph_rows]
panels = [
    ("loss", "Loss"), ("iou_score", "IoU"), ("fscore", "F1-score"),
    ("accuracy", "Accuracy"), ("precision", "Precision"), ("recall", "Recall"),
]


def draw_metric_panels(items, filename, columns, show=False):
    rows = (len(items) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(5.3 * columns, 4 * rows), squeeze=False)
    for ax, (key, title) in zip(axes.flat, items):
        ax.plot(graph_epochs, [float(row[key]) for row in graph_rows], label="Train")
        ax.plot(graph_epochs, [float(row["normal_val_" + key]) for row in graph_rows], label="Normal validation")
        ax.plot(graph_epochs, [float(row["val_" + key]) for row in graph_rows], label="Calibrated validation")
        ax.set(title=title, xlabel="Epoch", ylabel=title)
        ax.grid(True, alpha=0.3)
        ax.legend()
    for ax in list(axes.flat)[len(items):]:
        ax.set_visible(False)
    fig.suptitle(f"{EXPERIMENT_NAME} - SegFormer (calibrated)")
    fig.tight_layout()
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=180, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)
    print("Saved:", path)


draw_metric_panels(panels, "metric_graph.png", 3, show=True)
draw_metric_panels(panels[:2], "metric_graph_loss_iou.png", 2)
draw_metric_panels(panels[2:4], "metric_graph_f1_acc.png", 2)
draw_metric_panels(panels[4:], "metric_graph_pre_recall.png", 2)


In [ ]:
# Define test and prediction export

def cnn_confusion_image(target, prediction):
    # Preserve the reference CNN confusion-map colors.
    output = np.zeros((*target.shape, 3), dtype=np.uint8)
    output[(target == 0) & (prediction == 0)] = (0, 255, 0)
    output[(target == 0) & (prediction == 1)] = (0, 0, 255)
    output[(target == 1) & (prediction == 0)] = (255, 0, 0)
    return output


def save_prediction_visual(name, image_rgb, target, prediction, confusion, metrics):
    path = PREDICTION_DIR / (Path(name).stem + ".png")
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
    axes[0].imshow(image_rgb)
    axes[1].imshow(target, cmap="gray", vmin=0, vmax=1)
    axes[2].imshow(prediction, cmap="gray", vmin=0, vmax=1)
    axes[3].imshow(confusion)
    for ax, title in zip(axes, ("Image", "Ground truth", "Prediction", "Confusion (CNN palette)")):
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(f"{name} | IoU={metrics['iou']:.4f} | F1={metrics['f1']:.4f}")
    fig.tight_layout()
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)


def evaluate_and_export_test():
    if not FINAL_PATH.is_file() or not BEST_PATH.is_file():
        raise RuntimeError("Complete the 50-epoch training run first.")
    saved = torch.load(BEST_PATH, map_location="cpu", weights_only=True)
    if saved.get("bn_calibrated") is not True or saved.get("experiment") != EXPERIMENT_NAME:
        raise RuntimeError("A calibrated checkpoint from the selected experiment is required.")
    final = torch.load(FINAL_PATH, map_location="cpu", weights_only=True)
    if final["epoch"] != EPOCHS:
        raise RuntimeError("The final checkpoint does not represent the complete 50-epoch run.")
    del final
    dataset = DATASETS[EXPERIMENT_NAME]["Test"]
    stems = [Path(name).stem for name in dataset.names]
    if len(stems) != len(set(stems)):
        raise RuntimeError("Test filenames contain duplicate stems.")
    summary_path = OUTPUT_DIR / "test_summary.json"
    fingerprint = checkpoint_hash(BEST_PATH)
    if summary_path.exists():
        previous = json.loads(summary_path.read_text(encoding="utf-8"))
        if previous.get("checkpoint_sha256") != fingerprint:
            raise RuntimeError("The saved test results belong to another checkpoint and will not be overwritten.")
        outputs_complete = all(
            (PREDICTION_DIR / f"{stem}.png").is_file()
            and (MASK_DIR / f"{stem}.tif").is_file()
            and (CONFUSION_DIR / f"{stem}.png").is_file()
            for stem in stems
        )
        if not outputs_complete:
            raise RuntimeError("Test results exist, but some prediction files are missing.")
        print("Test results and predictions already exist; inference will not be repeated.")
        return previous
    for path in (PREDICTION_DIR, MASK_DIR, CONFUSION_DIR):
        path.mkdir(parents=True, exist_ok=True)
    loader = DataLoader(dataset, batch_size=1, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_model = SegFormerBinary().to(DEVICE)
    test_model.load_state_dict(saved["model_state_dict"], strict=True)
    test_model.eval()
    rows = []
    per_patch_path = OUTPUT_DIR / "test_per_patch.csv"
    try:
        with per_patch_path.open("w", newline="", encoding="utf-8") as file:
            writer = None
            with torch.no_grad():
                for index, (images, masks) in enumerate(loader):
                    images = images.to(DEVICE, non_blocking=True)
                    masks = masks.to(DEVICE, non_blocking=True)
                    logits = test_model(images)
                    values = {name: value[0].item() for name, value in patch_metrics(logits, masks).items()}
                    row = {"filename": dataset.names[index],
                           "loss": dice_loss_with_logits(logits, masks).item(), **values}
                    target = (masks[0, 0].cpu().numpy() >= 0.5).astype(np.uint8)
                    prediction = (torch.sigmoid(logits)[0, 0].cpu().numpy() >= THRESHOLD).astype(np.uint8)
                    row.update(
                        tp=int(np.sum((target == 1) & (prediction == 1))),
                        fp=int(np.sum((target == 0) & (prediction == 1))),
                        fn=int(np.sum((target == 1) & (prediction == 0))),
                        tn=int(np.sum((target == 0) & (prediction == 0))),
                    )
                    name = dataset.names[index]
                    stem = Path(name).stem
                    image_bgr = cv2.imread(str(dataset.images[name]), cv2.IMREAD_COLOR)
                    if image_bgr is None:
                        raise RuntimeError(f"Cannot read image: {dataset.images[name]}")
                    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
                    confusion = cnn_confusion_image(target, prediction)
                    if not cv2.imwrite(str(MASK_DIR / f"{stem}.tif"), prediction):
                        raise RuntimeError(f"Cannot save prediction mask: {name}")
                    if not cv2.imwrite(str(CONFUSION_DIR / f"{stem}.png"),
                                       cv2.cvtColor(confusion, cv2.COLOR_RGB2BGR)):
                        raise RuntimeError(f"Cannot save confusion map: {name}")
                    save_prediction_visual(name, image_rgb, target, prediction, confusion, values)
                    if writer is None:
                        writer = csv.DictWriter(file, fieldnames=list(row))
                        writer.writeheader()
                    writer.writerow(row)
                    file.flush()
                    rows.append(row)
                    if index == 0 or (index + 1) % 25 == 0 or index + 1 == len(dataset):
                        print(f"Test/prediction: {index + 1}/{len(dataset)}")
    finally:
        del test_model
        torch.cuda.empty_cache()
    if len(rows) != len(dataset):
        raise RuntimeError("Not all test patches were processed.")
    summary = {
        "experiment": EXPERIMENT_NAME, "selected_epoch": saved["epoch"],
        "checkpoint": str(BEST_PATH), "checkpoint_sha256": fingerprint,
        "bn_calibrated": True, "test_patches": len(rows),
        "test": {key: float(np.mean([row[key] for row in rows]))
                 for key in ("loss", *METRIC_NAMES)},
    }
    save_json(summary_path, summary)
    write_rows(OUTPUT_DIR / "test_summary.csv", [{
        "experiment": EXPERIMENT_NAME, "selected_epoch": saved["epoch"],
        "test_patches": len(rows), **summary["test"],
    }])
    return summary


In [ ]:
# Evaluate the best checkpoint

test_summary = evaluate_and_export_test()
print("Epoch selected by validation:", test_summary["selected_epoch"])
print("Test patches:", test_summary["test_patches"])
for name, value in test_summary["test"].items():
    print(f"{name}: {value:.4f}")
print("Test summary:", OUTPUT_DIR / "test_summary.json")
print("Prediction visualizations:", PREDICTION_DIR)
print("Raw binary masks (0/1):", MASK_DIR)
print("Confusion:", CONFUSION_DIR)


In [ ]:
# Verify saved outputs

required_files = [BEST_PATH, FINAL_PATH, HISTORY_PATH, TRAIN_CSV_PATH,
                  OUTPUT_DIR / "config.json", OUTPUT_DIR / "metric_graph.png",
                  OUTPUT_DIR / "metric_graph_loss_iou.png",
                  OUTPUT_DIR / "metric_graph_f1_acc.png",
                  OUTPUT_DIR / "metric_graph_pre_recall.png",
                  OUTPUT_DIR / "test_per_patch.csv", OUTPUT_DIR / "test_summary.json",
                  OUTPUT_DIR / "test_summary.csv"]
missing = [str(path) for path in required_files if not path.is_file()]
if SAVE_EVERY_EPOCH:
    missing.extend(str(EPOCH_WEIGHTS_DIR / f"epoch_{epoch:03d}.pth")
                   for epoch in range(1, EPOCHS + 1)
                   if not (EPOCH_WEIGHTS_DIR / f"epoch_{epoch:03d}.pth").is_file())
test_names = DATASETS[EXPERIMENT_NAME]["Test"].names
for name in test_names:
    stem = Path(name).stem
    for path in (PREDICTION_DIR / f"{stem}.png", MASK_DIR / f"{stem}.tif",
                 CONFUSION_DIR / f"{stem}.png"):
        if not path.is_file():
            missing.append(str(path))
if missing:
    raise RuntimeError("Missing files:\n" + "\n".join(missing[:15]))
print("All checkpoint, metric, graph, and prediction files are present.")
print("Test files per visualization type:", len(test_names))
print("Output:", OUTPUT_DIR)
